# ARC-AGI-3 SG Submission (official-shape)

Mirrors the official `inversion/arc3-sample-submission-stochastic-goose` (LB 0.25) structure exactly:
1. install arc-agi from competition wheels
2. write our agent to `/kaggle/working/my_agent.py`
3. on RERUN: wait for gateway, set up `ARC-AGI-3-Agents` repo, run `main.py --agent myagent`
4. Edit-mode: write dummy submission.parquet

Diff from sample: our `MyAgent` has segment-prior + static-mask multiplied into coord sampling. Logic otherwise identical.

## 1. Install arc-agi SDK from competition wheels

In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

## 2. Bring our repo in + copy agent to /kaggle/working/my_agent.py

Avoids inline-rewriting the agent file each time — single source of truth lives in `src/agents/sg_kaggle_agent.py`.

In [ ]:
import os, sys, shutil, glob

REPO_DATASET_CANDIDATES = [
    "/kaggle/input/arc-agi-3-pre",
    "/kaggle/input/arc-agi-3-pre-repo",
    "/kaggle/input/arc-agi-3-pre-bfs",
    "/kaggle/input/arc-agi-3-pre-v1",
    "/kaggle/input/arc-agi-3-pre-code",
]
if not any(os.path.isdir(p) for p in REPO_DATASET_CANDIDATES):
    for cand in glob.glob("/kaggle/input/*"):
        if os.path.isfile(os.path.join(cand, "src/agents/sg_kaggle_agent.py")):
            REPO_DATASET_CANDIDATES.insert(0, cand)
            break

REPO_SRC = next((p for p in REPO_DATASET_CANDIDATES if os.path.isdir(p)), None)
REPO_DIR = "/kaggle/working/arc-agi-3-pre"

print("-- /kaggle/input contents --")
!ls /kaggle/input/ 2>/dev/null
print()

if REPO_SRC is None:
    raise SystemExit(
        f"Could not find our repo at any of {REPO_DATASET_CANDIDATES}. "
        f"Attach the dataset to this notebook."
    )
print(f"using repo source: {REPO_SRC}")
if not os.path.isdir(REPO_DIR):
    shutil.copytree(REPO_SRC, REPO_DIR)

# Copy our agent into the conventional submission location
shutil.copy(f"{REPO_DIR}/src/agents/sg_kaggle_agent.py", "/kaggle/working/my_agent.py")
print(f"wrote /kaggle/working/my_agent.py from {REPO_SRC}/src/agents/sg_kaggle_agent.py")

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"torch device = {DEVICE} ({torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'})")

## 3. Run via official ARC-AGI-3-Agents runner (rerun-mode only)

Direct copy of the inversion sample's cell 3. Only diff: we don't %%writefile the agent inline — cell 2 already put it at `/kaggle/working/my_agent.py`.

In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # 1) wait for gateway to come up
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    # 2) copy runner repo to writable location
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    # 3) install our agent into the runner's templates dir
    !cp /kaggle/working/my_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    # 4) rewrite agents/__init__.py (original eager-imports langgraph/smolagents which aren't installed)
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write("""from typing import Type, cast
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    \"random\": Random,
    \"myagent\": MyAgent,
}
""")

    # 5) write .env pointing at gateway (loaded second with override=True by main.py)
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
""")

    # 6) run
    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg python main.py --agent myagent
else:
    print("Not in competition rerun — skipping agent run. (Dummy parquet next cell.)")

## 4. Edit-mode dummy submission

In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import pandas as pd
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    print(submission)